# 🎮 CartPole DQN — Deep Q-Learning from Raw Pixels

### Reproducing the core idea of the **DQN** paper ([arXiv:1312.5602](https://arxiv.org/abs/1312.5602)) on a single free GPU

**Subtitle.** How a convolutional neural network, trained with a variant of
**Q-learning** plus an **experience-replay** memory, can learn a control policy
*directly from screen pixels* — the breakthrough that launched deep
reinforcement learning.

| | |
|---|---|
| **Hardware target** | Google Colab **free tier (T4, 16 GB)** — also runs CPU-only, just slower |
| **Core frameworks** | `PyTorch`, `gymnasium[classic-control]`, `numpy`, `matplotlib`, `imageio`, `Pillow` |
| **Prerequisites** | Basic reinforcement learning (MDPs, value functions), convolutional neural networks, and PyTorch |
| **Runtime** | ~10–20 min end-to-end on a T4 (most of it is the training loop) |

> 🎮 **For research and educational purposes only.** This notebook is a
> *didactic reconstruction* of the 2013 DQN paper, not the official benchmark.
> The paper trains on **7 Atari 2600 games for 10 million frames each** — hours
> to days of GPU time. To fit a free T4, we train the *same algorithm and the
> same CNN* on **CartPole rendered as pixels**, which converges in minutes while
> still learning control from raw images. Absolute numbers are seed- and
> hardware-dependent; the *pipeline and the learning dynamics* are the lesson.

## 1 · What you will learn

The thesis of the DQN paper we reproduce here:

> **A single neural network can learn control from raw pixels, end-to-end.**
> No hand-crafted features, no access to the emulator's internal state — just
> the video frames, the reward, and the set of actions. The two tricks that make
> it work are **experience replay** (to break the correlation between
> consecutive samples) and framing Q-learning as a **regression problem** solved
> by stochastic gradient descent.

| # | Milestone | What you implement |
|---|-----------|--------------------|
| 1 | RL setup, Bellman equation | the math behind $Q^\star$ |
| 2 | The Q-network & its loss | Eq. (2)/(3) as a PyTorch loss |
| 3 | Experience replay & $\epsilon$-greedy | a replay buffer + exploration schedule |
| 4 | Preprocessing $\phi$ | grayscale → 84×84 → **stack 4 frames** |
| 5 | The DQN convolutional network | the paper's exact 16→32→256 architecture |
| 6 | Deep Q-learning (**Algorithm 1**) | the full training loop |
| 7 | Results | reward curve + the smoother **average-Q** curve (Fig. 2) |
| 8 | Value over an episode | a value-function trace (Fig. 3) |
| 9 | Watch the agent | a rendered GIF of greedy play |
| 10 | Bonus / questions | target networks, real Atari, and more |

**Important note.** We implement the **2013 core** — Algorithm 1, experience
replay, $\epsilon$-greedy, RMSProp — and then add the **one** ingredient the
2015 *Nature* DQN introduced to make training reliably stable: a **target
network**. It is controlled by a single switch (`use_target_net`); flip it off to
recover the exact 2013 behaviour (and watch the Q-values diverge). We also
**crop** the frame onto the pole so the CNN gets a strong signal — details
below.

## 2 · Environment setup & global configuration

### 2.1 Hardware check

Confirm we have a GPU. On Colab: **Runtime → Change runtime type → T4 GPU**.
The notebook runs on CPU too — training is just slower.

In [ ]:
# ── Hardware verification ──
!nvidia-smi -L 2>/dev/null || echo "No GPU detected — training will run on CPU (slower but fine)."

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

### 2.2 Dependencies

`gymnasium` provides the CartPole environment; `Pillow` does the image
preprocessing; `imageio` assembles the final GIF. We set a *dummy* SDL video
driver so `pygame` (CartPole's renderer) produces frames **headlessly** on a
server with no display — the usual Colab situation.

In [ ]:
# ── Dependency installation (Colab) ──
# Quiet installs; versions left unpinned so the notebook keeps working over time.
!pip install -q "gymnasium[classic-control]" imageio imageio-ffmpeg pillow 2>/dev/null

# Headless rendering for pygame-based classic-control envs.
import os
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")
print("Dependencies ready; SDL video driver =", os.environ["SDL_VIDEODRIVER"])

### 2.3 Global imports, seed, plotting & device

One cell for every import, a global `SEED` for reproducibility, a consistent
plotting style, and the `DEVICE` we put tensors on.

In [ ]:
# ── Global configuration ──
import random, math, time
from collections import deque, namedtuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
import gymnasium as gym

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

print("Device:", DEVICE)

## 3 · Theory — from Q-learning to a Deep Q-Network

We follow the paper's derivation. The goal is a policy that maximises
cumulative reward; the tool is the **action-value function** $Q$, which we will
approximate with a CNN.

### 3.1 The RL setup: states, actions, rewards, return

An agent interacts with an environment $\mathcal{E}$ (the emulator) in a
sequence of **actions** $a_t$, **observations** $x_t$ (raw pixels), and scalar
**rewards** $r_t$. Because a single frame is not enough to understand motion,
the paper defines the *state* as the recent history of frames — which is exactly
why we will **stack 4 frames** in §4.

The agent maximises the **future discounted return**

$$R_t = \sum_{t'=t}^{T} \gamma^{\,t'-t}\, r_{t'},$$

where $\gamma \in [0,1)$ trades off immediate versus future reward and $T$ is the
terminal step. The **optimal action-value function** is the best expected return
achievable after seeing state $s$ and taking action $a$:

$$Q^\star(s,a) = \max_{\pi}\, \mathbb{E}\!\left[R_t \mid s_t=s,\, a_t=a,\, \pi\right].$$

### 3.2 The Bellman equation & value iteration

$Q^\star$ satisfies the **Bellman optimality equation** — if you knew the
optimal value of the next state $s'$ for every action $a'$, the optimal move is
the one maximising $r + \gamma\, Q^\star(s',a')$:

$$Q^\star(s,a) = \mathbb{E}_{s'\sim\mathcal{E}}\!\left[\, r + \gamma \max_{a'} Q^\star(s',a') \;\middle|\; s,a \right]. \tag{1}$$

Value iteration ($Q_{i+1} = \mathbb{E}[r + \gamma \max_{a'} Q_i(s',a')]$)
converges to $Q^\star$, but tabulating a value per state is hopeless for
images — there are astronomically many. We need a **function approximator**.

### 3.3 Function approximation: the Q-network & its loss

We approximate $Q^\star$ with a neural network of weights $\theta$, a
**Q-network** $Q(s,a;\theta)$. It is trained by minimising a sequence of
squared-error losses that turn the Bellman equation into **regression**:

$$L_i(\theta_i) = \mathbb{E}_{s,a\sim\rho(\cdot)}\!\left[\big(y_i - Q(s,a;\theta_i)\big)^2\right], \tag{2}$$

with the temporal-difference **target**

$$y_i = \mathbb{E}_{s'\sim\mathcal{E}}\!\left[\, r + \gamma \max_{a'} Q(s',a';\theta_{i-1}) \;\middle|\; s,a \right].$$

Differentiating gives the gradient the paper actually uses:

$$\nabla_{\theta_i} L_i(\theta_i) = \mathbb{E}\!\left[\big(r + \gamma \max_{a'} Q(s',a';\theta_{i-1}) - Q(s,a;\theta_i)\big)\, \nabla_{\theta_i} Q(s,a;\theta_i)\right]. \tag{3}$$

Two properties matter:

* **Model-free** — we never build a model of $\mathcal{E}$; we learn straight
  from samples.
* **Off-policy** — we learn about the *greedy* policy
  $a = \arg\max_a Q(s,a;\theta)$ while *behaving* with exploration.

> ⚠️ **2013 vs. 2015.** In this paper the target uses the network's own
> parameters (Algorithm 1 writes $\theta$, not a frozen $\theta^-$). Bootstrapping
> off the *same* rapidly-moving weights is unstable and makes the Q-values blow
> up. The 2015 fix is a **target network**: a periodically-frozen copy
> $\theta^-$ used only to form the target $y = r + \gamma \max_{a'} Q(s',a';\theta^-)$.
> Our training loop uses $\theta^-$ by default (`use_target_net=True`) and lets
> you switch back to the pure-2013 $\theta$ to see the difference.

### 3.4 Experience replay & $\epsilon$-greedy

Feeding a neural net the stream of consecutive frames breaks two deep-learning
assumptions: samples are **highly correlated**, and the data distribution
**shifts** as the policy changes — which can make training oscillate or diverge.
The paper's fix is **experience replay**:

1. Store each transition $e_t = (s_t, a_t, r_t, s_{t+1})$ in a memory
   $\mathcal{D}$ of the last $N$ transitions.
2. Apply Q-learning updates to **random minibatches** drawn from $\mathcal{D}$.

Random sampling breaks correlations and averages the behaviour distribution over
many past states, smoothing learning. Because we learn off-policy, replay is
sound (and it makes each experience reusable across many updates).

Exploration uses an **$\epsilon$-greedy** behaviour policy: act greedily with
probability $1-\epsilon$, act randomly with probability $\epsilon$. We
**anneal** $\epsilon$ linearly from $1.0$ to $0.1$, as in the paper.

### 3.5 Algorithm 1 at a glance

The whole method — **Deep Q-learning with Experience Replay**:

```
Initialize replay memory D to capacity N
Initialize action-value function Q with random weights θ
for episode = 1, M do
    Initialise state s₁ and preprocessed φ₁ = φ(s₁)
    for t = 1, T do
        With probability ε select a random action aₜ
        otherwise select aₜ = argmaxₐ Q(φ(sₜ), a; θ)
        Execute aₜ, observe reward rₜ and image xₜ₊₁
        Set sₜ₊₁ = sₜ, aₜ, xₜ₊₁ and preprocess φₜ₊₁ = φ(sₜ₊₁)
        Store transition (φₜ, aₜ, rₜ, φₜ₊₁) in D
        Sample random minibatch (φⱼ, aⱼ, rⱼ, φⱼ₊₁) from D
        Set yⱼ = rⱼ                              (for terminal φⱼ₊₁)
               = rⱼ + γ maxₐ' Q(φⱼ₊₁, a'; θ)     (otherwise)
        Do a gradient step on (yⱼ − Q(φⱼ, aⱼ; θ))²   (Eq. 3)
    end for
end for
```

Everything below is a direct implementation of this pseudocode — with the one
2015 addition (a target network for the $\max_{a'}$ term) that we motivated in
§3.3 and keep switchable.

## 4 · The environment & preprocessing $\phi$

### 4.1 CartPole as pixels

`CartPole-v1`: push a cart left/right to keep a pole balanced; **+1 reward per
timestep**, episode ends when the pole falls or after 500 steps. The agent
normally sees a 4-number state — but to reproduce the paper we throw that away
and learn from the **rendered image** instead (`render_mode="rgb_array"`).

The cell below builds the env, renders a frame, and — per the tutorial spec —
**falls back to a synthetic placeholder** if headless rendering is unavailable,
so the notebook always runs.

In [ ]:
# ── Build the environment and grab one raw RGB frame (with fallback) ──
def make_env():
    """Create CartPole-v1 configured to return RGB frames."""
    return gym.make("CartPole-v1", render_mode="rgb_array")

def safe_render(env):
    """Render an RGB frame, falling back to a synthetic placeholder.

    Headless boxes without a working display can fail to render. Rather than
    crash the whole notebook we return a grey placeholder with a moving marker
    so every downstream cell still executes.
    """
    try:
        frame = env.render()
        if frame is None:
            raise RuntimeError("env.render() returned None")
        return np.asarray(frame, dtype=np.uint8)
    except Exception as exc:  # pragma: no cover - depends on the host display
        print("⚠️  render failed (%s); using synthetic placeholder frame." % exc)
        ph = np.full((400, 600, 3), 220, dtype=np.uint8)
        x = int(300 + 120 * math.sin(time.time()))
        ph[180:260, x - 5:x + 5] = 40  # a dark 'pole' so preprocessing has signal
        return ph

env = make_env()
obs, info = env.reset(seed=SEED)
raw = safe_render(env)
print("Raw frame shape:", raw.shape, "| action space:", env.action_space)

plt.figure(figsize=(4, 3))
plt.imshow(raw)
plt.title("Raw rendered CartPole frame")
plt.axis("off")
plt.show()

### 4.2 $\phi$: grayscale → downsample → 84×84 → stack 4 frames

The paper's preprocessing $\phi$ (§4.1): convert to grayscale, downsample, and
crop to **84×84**, then **stack the last 4 frames** so the network can perceive
motion (velocity, here the pole's angular velocity).

The paper crops to "the playing area". Here that matters a lot: the CartPole
render is `400×600` but the cart and pole occupy only a central band
(rows ≈150–330 — the rest is empty sky and ground). If we resized the *whole*
frame the pole would shrink to a few faint pixels and the network would barely
see it. So we **crop to that band first**, which makes the pole a tall, prominent
feature in the 84×84 input — a small change that is decisive for learning.

* `preprocess(frame)` → a single `uint8` 84×84 grayscale image (cropped).
* `FrameStack` maintains a rolling buffer of the last 4 processed frames and
  returns the `(4, 84, 84)` state $\phi$.

In [ ]:
# ── Preprocessing φ: crop → grayscale → downsample + 4-frame stacking ──
FRAME_SIZE = 84
STACK = 4
# The cart+pole live in rows ≈150–330 of the 400×600 render; crop to that band
# (full width, so the cart is never clipped) before resizing so the pole fills
# the 84×84 input instead of washing out.
CROP_TOP, CROP_BOTTOM = 150, 330

def preprocess(frame):
    """RGB frame -> uint8 grayscale 84x84 image: crop to the pole band, then φ."""
    frame = frame[CROP_TOP:CROP_BOTTOM, :]          # zoom onto cart + pole
    img = Image.fromarray(frame).convert("L").resize(
        (FRAME_SIZE, FRAME_SIZE), Image.BILINEAR)
    return np.asarray(img, dtype=np.uint8)

class FrameStack:
    """Rolling buffer of the last STACK processed frames -> φ of shape (STACK,84,84)."""
    def __init__(self, k=STACK):
        self.k = k
        self.frames = deque(maxlen=k)

    def reset(self, frame):
        f = preprocess(frame)
        for _ in range(self.k):
            self.frames.append(f)      # duplicate the first frame to fill the stack
        return self._state()

    def step(self, frame):
        self.frames.append(preprocess(frame))
        return self._state()

    def _state(self):
        return np.stack(self.frames, axis=0)  # (k, 84, 84) uint8

# Visualise the 4 channels of a freshly reset stack.
stacker = FrameStack()
state = stacker.reset(raw)
print("Stacked state φ shape:", state.shape, state.dtype)

fig, axes = plt.subplots(1, STACK, figsize=(10, 3))
for i, ax in enumerate(axes):
    ax.imshow(state[i], cmap="gray")
    ax.set_title(f"frame t-{STACK-1-i}")
    ax.axis("off")
plt.suptitle("φ = 4 stacked 84×84 grayscale frames")
plt.show()

## 5 · The Deep Q-Network

The paper's exact architecture (§4, "Model Architecture"):

| Layer | Spec | Output |
|-------|------|--------|
| Input | stacked frames | `4 × 84 × 84` |
| Conv 1 | 16 filters, 8×8, stride 4, ReLU | `16 × 20 × 20` |
| Conv 2 | 32 filters, 4×4, stride 2, ReLU | `32 × 9 × 9` |
| FC | 256 units, ReLU | `256` |
| Output | linear, one per action | `n_actions` |

Crucially the network takes **only the state** as input and outputs a Q-value
for **every action at once** — so a single forward pass evaluates all actions.

In [ ]:
# ── The DQN convolutional network ──
class DQN(nn.Module):
    """Convolutional Q-network: φ (4×84×84) -> one Q-value per action.

    Matches the 2013 paper: 16 8×8/4 conv, 32 4×4/2 conv, 256-unit FC, then a
    linear output head with one unit per valid action.
    """
    def __init__(self, n_actions, in_channels=STACK):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 16, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=4, stride=2)
        self.fc    = nn.Linear(32 * 9 * 9, 256)
        self.head  = nn.Linear(256, n_actions)

    def forward(self, x):
        # x: (B, 4, 84, 84) float in [0, 1]
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.fc(x.flatten(1)))
        return self.head(x)

n_actions = env.action_space.n
net = DQN(n_actions).to(DEVICE)
n_params = sum(p.numel() for p in net.parameters())
print(net)
print(f"\nValid actions: {n_actions} | trainable parameters: {n_params:,}")

# Sanity check: a forward pass on one state produces one Q-value per action.
with torch.no_grad():
    dummy = torch.as_tensor(state[None], dtype=torch.float32, device=DEVICE) / 255.0
    print("Q(φ, ·) =", net(dummy).cpu().numpy().round(3))

## 6 · Replay memory & $\epsilon$-greedy policy

**Replay memory** stores the last $N$ transitions and returns uniform random
minibatches. We keep states as `uint8` (0–255) to save memory — a stack is
`4×84×84 = 28 KB`, so 20 000 transitions with next-states is ~1.1 GB, well
within a T4. We convert to float and scale by `1/255` only inside the update.

**$\epsilon$-greedy** picks a random action with probability $\epsilon$, else the
greedy $\arg\max_a Q$. $\epsilon$ anneals linearly $1.0 \to 0.1$ over the first
chunk of training, then stays at $0.1$ (paper §5).

In [ ]:
# ── Replay memory (ring buffer) ──
Transition = namedtuple("Transition", ("state", "action", "reward", "next_state", "done"))

class ReplayMemory:
    """Fixed-capacity buffer of transitions with uniform random sampling."""
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, *args):
        """Store one transition (state, action, reward, next_state, done)."""
        self.buffer.append(Transition(*args))

    def sample(self, batch_size):
        """Return a random minibatch as stacked tensors on DEVICE."""
        batch = random.sample(self.buffer, batch_size)
        b = Transition(*zip(*batch))
        # states are uint8 (B,4,84,84); scale to [0,1] floats for the network.
        s  = torch.as_tensor(np.stack(b.state),      dtype=torch.float32, device=DEVICE) / 255.0
        ns = torch.as_tensor(np.stack(b.next_state), dtype=torch.float32, device=DEVICE) / 255.0
        a  = torch.as_tensor(b.action, dtype=torch.int64,   device=DEVICE).unsqueeze(1)
        r  = torch.as_tensor(b.reward, dtype=torch.float32, device=DEVICE).unsqueeze(1)
        d  = torch.as_tensor(b.done,   dtype=torch.float32, device=DEVICE).unsqueeze(1)
        return s, a, r, ns, d

    def __len__(self):
        return len(self.buffer)


# ── ε-greedy action selection ──
def epsilon_by_step(step, eps_start=1.0, eps_end=0.1, eps_decay_steps=10_000):
    """Linear annealing of ε from eps_start to eps_end over eps_decay_steps."""
    frac = min(1.0, step / eps_decay_steps)
    return eps_start + frac * (eps_end - eps_start)

def select_action(state, eps, policy_net, n_actions):
    """ε-greedy: random action w.p. ε, else argmax_a Q(state, a; θ)."""
    if random.random() < eps:
        return random.randrange(n_actions)
    with torch.no_grad():
        s = torch.as_tensor(state[None], dtype=torch.float32, device=DEVICE) / 255.0
        return int(policy_net(s).argmax(dim=1).item())

print("ReplayMemory and ε-greedy policy ready.")
print("ε at steps [0, 5k, 10k, 20k]:",
      [round(epsilon_by_step(s), 3) for s in (0, 5_000, 10_000, 20_000)])

## 7 · Training: Deep Q-learning with Experience Replay (Algorithm 1)

We now assemble Algorithm 1. Faithful details from the paper:

* **RMSProp** optimiser, minibatches of **32**.
* **Reward clipping** to $\{-1, 0, +1\}$. *(For CartPole every step gives $+1$,
  so clipping is a no-op here — we keep it for fidelity and so the same loop
  works on Atari.)*
* **TD target**, detached: $y_j = r_j$ if $s_{j+1}$ is terminal, else
  $r_j + \gamma \max_{a'} Q(s_{j+1},a';\theta^-)$. By default $\theta^-$ is a
  **target network** (synced from $\theta$ every `target_update` steps) — the
  2015 stability fix. Set `use_target_net=False` for the pure-2013 target that
  uses the live $\theta$.
* **A cropped, pole-filling input** (§4.2) so the CNN actually sees the pole.
* We track two metrics from paper §5.1: the noisy **reward per episode**, and the
  much smoother **average max-predicted-Q on a fixed held-out set of states**.

> ⏱️ **Free-tier budget & disconnects.** Colab free sessions can drop mid-run.
> This loop **checkpoints every `checkpoint_every` steps and resumes
> automatically** — if the runtime disconnects, just **re-run this cell** and it
> continues from the last checkpoint. `total_steps` is a *cumulative target*, so
> re-running never restarts from zero (raise it for a stronger agent). Pass
> `resume=False`, or delete `CKPT_PATH`, to train from scratch. To survive a
> *full* runtime reset, mount Google Drive and point `CKPT_PATH` at a folder
> there:
>
> ```python
> from google.colab import drive; drive.mount("/content/drive")
> CKPT_PATH = "/content/drive/MyDrive/dqn_cartpole_ckpt.pt"
> ```

**A subtlety — time limits vs. real terminals.** CartPole ends either because the
pole fell (`terminated`) *or* because it hit the 500-step limit (`truncated`).
Only a *true* terminal should zero the bootstrap; on a time-limit truncation the
episode was still "going", so we keep bootstrapping. We therefore set the
`done` flag used in the target from `terminated` **only**.

In [ ]:
# ── Hyperparameters (Algorithm 1 + the 2015 target-network fix) ──
# `total_steps` is a CUMULATIVE target: training resumes from a checkpoint and
# runs until the global step count reaches it. If Colab disconnects, just re-run
# the training cell — it continues from the last checkpoint (see §7).
CKPT_PATH = "dqn_cartpole_ckpt.pt"   # set to a Google Drive path to survive full runtime resets

CONFIG = dict(
    total_steps      = 60_000,   # cumulative target; re-run the cell to keep going
    replay_capacity  = 20_000,
    batch_size       = 32,
    gamma            = 0.99,
    lr               = 2.5e-4,   # RMSProp; the target network keeps this stable and learns faster
    learning_starts  = 1_000,    # fill the buffer a bit before training
    train_every      = 1,        # gradient step every N env steps
    eps_start        = 1.0,
    eps_end          = 0.1,
    eps_decay_steps  = 15_000,   # explore for the first ~quarter of training
    eval_every       = 2_000,    # how often to log the held-out average Q
    checkpoint_every = 5_000,    # save every N steps so a disconnect costs at most this many
    use_target_net   = True,     # True = 2015 stability fix; False = pure-2013 (unstable)
    target_update    = 1_000,    # sync θ⁻ ← θ every N steps (ignored if use_target_net=False)
)
CONFIG

In [ ]:
# ── Collect a fixed held-out set of states (paper §5.1) ──
# Run a random policy before training and freeze ~500 states; the average of
# max-predicted-Q over these is the *smooth* progress metric.
def collect_holdout(n=500):
    """Roll out a random policy and return a stack of held-out states."""
    e = make_env()
    e.reset(seed=SEED + 123)
    stk = FrameStack()
    stk.reset(safe_render(e))
    states = []
    while len(states) < n:
        _, _, term, trunc, _ = e.step(e.action_space.sample())
        states.append(stk.step(safe_render(e)))
        if term or trunc:
            e.reset()
            stk.reset(safe_render(e))
    e.close()
    return torch.as_tensor(np.stack(states), dtype=torch.float32, device=DEVICE) / 255.0

holdout_states = collect_holdout()
print("Held-out states:", tuple(holdout_states.shape))

@torch.no_grad()
def average_holdout_q(policy_net):
    """Mean over held-out states of max_a Q(s, a) — the smooth metric."""
    return policy_net(holdout_states).max(dim=1).values.mean().item()

In [ ]:
# ── Checkpointing so training survives Colab disconnects ──
def save_checkpoint(path, policy_net, target_net, optimizer, global_step, logs):
    """Persist weights, optimiser state, step count and logs to `path`."""
    torch.save({
        "policy": policy_net.state_dict(),
        "target": target_net.state_dict(),
        "optim":  optimizer.state_dict(),
        "global_step": global_step,
        "logs": logs,
    }, path)


# ── The deep Q-learning training loop (resumable) ──
def train_dqn(cfg, resume=True, ckpt_path=CKPT_PATH):
    """Run Algorithm 1 up to a cumulative `total_steps`, checkpointing as we go.

    `total_steps` is a *target*: the loop resumes from `ckpt_path` (if present and
    resume=True) and runs until the global step count reaches it. If the runtime
    disconnects, just re-run this cell — it continues from the last checkpoint.
    Pass resume=False (or delete the checkpoint file) to train from scratch.
    """
    env = make_env()
    n_actions = env.action_space.n

    policy_net = DQN(n_actions).to(DEVICE)          # θ  (online network)
    target_net = DQN(n_actions).to(DEVICE)          # θ⁻ (2015 target network)
    optimizer  = torch.optim.RMSprop(policy_net.parameters(), lr=cfg["lr"])
    logs = dict(ep_returns=[], q_curve=[], loss_curve=[])
    global_step = 0

    if resume and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=DEVICE)
        policy_net.load_state_dict(ck["policy"])
        target_net.load_state_dict(ck["target"])
        optimizer.load_state_dict(ck["optim"])
        global_step, logs = ck["global_step"], ck["logs"]
        print(f"↩︎  Resumed from {ckpt_path} at global step {global_step}.")
    else:
        target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()
    target_src = target_net if cfg["use_target_net"] else policy_net  # θ⁻ or live θ

    remaining = cfg["total_steps"] - global_step
    if remaining <= 0:
        print(f"✓ Target of {cfg['total_steps']} steps already reached — nothing to do.")
        env.close()
        return policy_net, logs

    # The replay buffer is rebuilt each run (too big to checkpoint); it refills
    # in ~learning_starts steps, after which gradient updates resume.
    memory = ReplayMemory(cfg["replay_capacity"])
    stacker = FrameStack()
    env.reset(seed=SEED + global_step)
    state = stacker.reset(safe_render(env))
    ep_return = 0.0
    t0 = time.time()

    try:
        for _ in range(remaining):
            global_step += 1
            eps = epsilon_by_step(global_step, cfg["eps_start"], cfg["eps_end"], cfg["eps_decay_steps"])
            action = select_action(state, eps, policy_net, n_actions)

            _, reward, terminated, truncated, _ = env.step(action)
            next_state = stacker.step(safe_render(env))
            ep_return += reward
            reward = float(np.clip(reward, -1, 1))              # reward clipping
            done_for_target = float(terminated)                 # truncation still bootstraps

            memory.push(state, action, reward, next_state, done_for_target)
            state = next_state

            if terminated or truncated:
                logs["ep_returns"].append(ep_return)
                ep_return = 0.0
                env.reset()
                state = stacker.reset(safe_render(env))

            # ── learn: one gradient step on a random minibatch ──
            if len(memory) >= cfg["learning_starts"] and global_step % cfg["train_every"] == 0:
                s, a, r, ns, d = memory.sample(cfg["batch_size"])
                q = policy_net(s).gather(1, a)                  # Q(φⱼ, aⱼ; θ)
                with torch.no_grad():                           # target detached from the graph
                    max_next = target_src(ns).max(dim=1, keepdim=True).values  # uses θ⁻ (or θ)
                    y = r + cfg["gamma"] * max_next * (1.0 - d) # yⱼ (Eq. 2)
                loss = F.smooth_l1_loss(q, y)                   # Huber ≈ Eq. (3), robust
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_value_(policy_net.parameters(), 1.0)
                optimizer.step()
                logs["loss_curve"].append((global_step, loss.item()))

            # ── sync the target network θ⁻ ← θ (2015 fix) ──
            if cfg["use_target_net"] and global_step % cfg["target_update"] == 0:
                target_net.load_state_dict(policy_net.state_dict())

            if global_step % cfg["eval_every"] == 0:
                avg_q = average_holdout_q(policy_net)
                logs["q_curve"].append((global_step, avg_q))
                recent = np.mean(logs["ep_returns"][-20:]) if logs["ep_returns"] else 0.0
                print(f"step {global_step:>6} | ε={eps:0.2f} | avg return(20)={recent:6.1f} "
                      f"| avg Q={avg_q:6.3f} | {time.time()-t0:5.0f}s")

            if global_step % cfg["checkpoint_every"] == 0:
                save_checkpoint(ckpt_path, policy_net, target_net, optimizer, global_step, logs)
    except KeyboardInterrupt:
        print("⏹  Interrupted — saving a checkpoint so you can resume later.")

    save_checkpoint(ckpt_path, policy_net, target_net, optimizer, global_step, logs)
    env.close()
    print(f"Saved checkpoint at global step {global_step} → {ckpt_path}")
    return policy_net, logs

policy_net, logs = train_dqn(CONFIG)
print("Training done. Total episodes so far:", len(logs["ep_returns"]))

## 8 · Results & visualization

### 8.1 Learning curves

The paper's Figure 2 makes a key point: **reward per episode is noisy**, but the
**average predicted Q** rises **smoothly** — evidence the network is steadily
improving even when the reward signal looks jumpy. We reproduce both.

In [ ]:
# ── Reward-per-episode and average-Q curves (à la Fig. 2) ──
returns = logs["ep_returns"]
q_steps = [s for s, _ in logs["q_curve"]]
q_vals  = [v for _, v in logs["q_curve"]]

def moving_avg(x, w=20):
    if len(x) < 1:
        return np.array([])
    w = min(w, len(x))
    return np.convolve(x, np.ones(w) / w, mode="valid")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(returns, alpha=0.35, label="per episode")
if len(returns) >= 1:
    ax1.plot(range(len(moving_avg(returns))), moving_avg(returns),
             lw=2, label="20-ep moving avg")
ax1.set_xlabel("episode"); ax1.set_ylabel("total reward")
ax1.set_title("Reward per episode (noisy)"); ax1.legend()

ax2.plot(q_steps, q_vals, "o-", lw=2, color="tab:green")
ax2.set_xlabel("training step"); ax2.set_ylabel("avg max predicted Q")
ax2.set_title("Average Q on held-out states (smooth)")

plt.tight_layout(); plt.show()

### 8.2 Value function over an episode

Figure 3 of the paper traces the predicted value while the game is played,
showing it rise and fall with the situation. We run one greedy episode and plot
$V(s_t) = \max_a Q(s_t, a)$ over time — for CartPole the value reflects "how
safe is the pole right now / how much more reward can I expect".

In [ ]:
# ── Trace V(sₜ)=maxₐ Q over one greedy episode (à la Fig. 3) ──
@torch.no_grad()
def value_trace(policy_net, max_steps=500):
    """Play one greedy episode, returning per-step value and a few frames."""
    e = make_env(); e.reset(seed=SEED + 7)
    stk = FrameStack(); s = stk.reset(safe_render(e))
    values, frames = [], []
    for t in range(max_steps):
        st = torch.as_tensor(s[None], dtype=torch.float32, device=DEVICE) / 255.0
        q = policy_net(st)
        values.append(q.max().item())
        if t % max(1, max_steps // 3) == 0:
            frames.append(safe_render(e))
        a = int(q.argmax().item())
        _, _, term, trunc, _ = e.step(a)
        s = stk.step(safe_render(e))
        if term or trunc:
            break
    e.close()
    return values, frames

values, frames = value_trace(policy_net)

fig = plt.figure(figsize=(12, 3.5))
gs = fig.add_gridspec(1, 3 + len(frames))
axv = fig.add_subplot(gs[0, :3])
axv.plot(values, lw=2, color="tab:purple")
axv.set_xlabel("frame #"); axv.set_ylabel("V = maxₐ Q")
axv.set_title(f"Predicted value over one episode ({len(values)} steps survived)")
for i, fr in enumerate(frames):
    ax = fig.add_subplot(gs[0, 3 + i]); ax.imshow(fr); ax.axis("off")
    ax.set_title(f"t≈{i * (len(values)//max(1,len(frames)))}")
plt.tight_layout(); plt.show()

## 9 · Watch the trained agent

Roll out the greedy policy, capture the frames, and assemble a GIF so you can
*see* what the network learned to do from pixels alone.

In [ ]:
# ── Record a greedy rollout and display it as a GIF ──
import imageio
from IPython.display import Image as IPyImage, display

@torch.no_grad()
def record_episode(policy_net, path="dqn_cartpole.gif", max_steps=500):
    """Play one greedy episode, save the rendered frames as a GIF."""
    e = make_env(); e.reset(seed=SEED + 42)
    stk = FrameStack(); s = stk.reset(safe_render(e))
    frames = [safe_render(e)]
    for _ in range(max_steps):
        st = torch.as_tensor(s[None], dtype=torch.float32, device=DEVICE) / 255.0
        a = int(policy_net(st).argmax().item())
        _, _, term, trunc, _ = e.step(a)
        frames.append(safe_render(e))
        s = stk.step(frames[-1])
        if term or trunc:
            break
    e.close()
    imageio.mimsave(path, frames, fps=30)
    print(f"Saved {len(frames)} frames to {path}")
    return path

gif_path = record_episode(policy_net)
try:
    display(IPyImage(filename=gif_path))
except Exception as exc:
    print("Could not inline the GIF (%s). Open the file directly: %s" % (exc, gif_path))

## 10 · Bonus & discussion questions

### Questions to think about

1. **Replay capacity.** Re-run training with `replay_capacity` set to `2_000`
   vs `50_000`. How does the size of the memory affect stability and the final
   reward? Why does *too small* a buffer reintroduce the correlation problem?
2. **The $\epsilon$ schedule.** What happens if you anneal $\epsilon$ to `0.0`
   instead of `0.1`, or decay it over `1_000` steps instead of `10_000`? Relate
   your observation to the exploration–exploitation trade-off.
3. **Why is average-Q smoother than reward?** Section 8.1 shows the reward curve
   is jumpy while the held-out Q rises smoothly. What does that tell you about
   using episode reward vs. predicted value to *monitor* RL training?
4. **What does reward clipping cost?** The paper clips all rewards to
   $\{-1,0,+1\}$. On CartPole this is a no-op — but on a game like Pong it throws
   away reward *magnitude*. When would that hurt, and what does it buy you?

### Going further

* **See the 2013 instability yourself.** Re-run §7 with `use_target_net=False`.
  Without the frozen $\theta^-$, the target chases the live weights: watch the
  average-Q curve blow up and the reward stall — exactly why the 2015 target
  network matters.
* **Double DQN.** Decouple action *selection* (online net) from *evaluation*
  (target net) to reduce Q-value overestimation.
* **Prioritized experience replay.** The paper itself suggests sampling
  "transitions from which we can learn the most" instead of uniformly.
* **Repoint at real Atari.** Install `gymnasium[atari,accept-rom-license]`, swap
  `make_env()` for e.g. `ALE/Pong-v5`, keep the *same* $\phi$ and CNN, and note
  what changes: a larger action space, frame-skip $k=4$, and **far** more
  training (millions of frames) to win.

---
*Built as a didactic reconstruction of **Playing Atari with Deep Reinforcement
Learning** (Mnih, Kavukcuoglu, Silver, Graves, Antonoglou, Wierstra & Riedmiller,
DeepMind, arXiv:1312.5602). For research & education only.*